# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "/workspace/model-organisms/diffing_results/olmo2_1B/italian_food_narrow-sft-0/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [2]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: /workspace/model-organisms/diffing_results/olmo2_1B/italian_food_narrow-sft-0/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 14 dir: /workspace/model-organisms/diffing_results/olmo2_1B/italian_food_narrow-sft-0/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 15 dir: /workspace/model-organisms/diffing_results/olmo2_1B/italian_food_narrow-sft-0/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [3]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                                    \
                       base             base_inv                           ft   
0           .Today (0.0240)      urrenc (0.0217)           Buccane (5.22e-03)   
1          .Second (0.0114)       pos (5.16e-03)             oller (4.76e-03)   
2        Buccane (8.85e-03)       act (4.85e-03)            .Today (3.60e-03)   
3          /Area (6.07e-03)    askell (3.54e-03)              oire (3.28e-03)   
4            .au (4.88e-03)      gons (3.33e-03)                ít (3.17e-03)   
5      /problems (4.03e-03)        �� (2.01e-03)       persistence (2.62e-03)   
6            aru (3.91e-03)      ejec (2.01e-03)               ERM (2.47e-03)   
7      /entities (2.96e-03)        دي (1.95e-03)   notwithstanding (2.47e-03)   
8          /bind (2.69e-03)      azon (1.95e-03)                ,a (2.24e-03)   
9       /problem (2.61e-03)     fácil (1.79e-03)         /problems (2.18e-03)   
10         /Math (2.61e-03)      anth (1.73e-03)           broadly (2.12e-03)   
11      /respond (2.61e-03)     posix (1.73e-03)             .Last (2.12e-03)   
12          fter (2.46e-03)  essional (1.57e-03)            wouldn (2.04e-03)   
13    confidence (2.30e-03)  Optional (1.53e-03)              itte (1.98e-03)   
14     /operator (2.23e-03)    Phones (1.48e-03)         belonging (1.87e-03)   
15   persistence (2.23e-03)      Vers (1.48e-03)              ifer (1.81e-03)   
16     /activity (2.17e-03)        vs (1.39e-03)              .Abs (1.75e-03)   
17     belonging (1.97e-03)      orst (1.27e-03)         /activity (1.75e-03)   
18          ilot (1.97e-03)  >Welcome (1.27e-03)             Fence (1.75e-03)   
19     .AddRange (1.85e-03)       med (1.23e-03)          concrete (1.75e-03)   

                                                                       \
                 ft_inv                 diff                 diff_inv   
0       urrenc (0.0103)         ach (0.0275)             neh (0.0374)   
1    ategori (4.55e-03)         lim (0.0107)           ource (0.0310)   
2     askell (4.27e-03)       ced (9.22e-03)        Uploader (0.0156)   
3       spos (4.03e-03)      amar (7.87e-03)              år (0.0129)   
4      posix (3.77e-03)    master (6.32e-03)       normals (5.07e-03)   
5         دي (3.54e-03)       dar (4.79e-03)        istema (3.94e-03)   
6      fácil (3.33e-03)      Prof (4.64e-03)       ---\n\n (3.27e-03)   
7       gons (3.13e-03)     domin (3.85e-03)          vang (3.07e-03)   
8      mostr (2.59e-03)    Modern (3.72e-03)          Lage (2.88e-03)   
9        pos (2.15e-03)         r (3.72e-03)           ICA (2.88e-03)   
10       ={< (1.67e-03)    lients (3.49e-03)        PECIAL (2.55e-03)   
11      moda (1.67e-03)      Stop (3.08e-03)       Farrell (2.55e-03)   
12     antha (1.67e-03)    System (3.08e-03)   cellspacing (2.55e-03)   
13  essional (1.57e-03)        ln (2.90e-03)         conta (2.55e-03)   
14         د (1.48e-03)       100 (2.90e-03)          holm (2.40e-03)   
15        zg (1.48e-03)      kill (2.72e-03)        mostra (2.40e-03)   
16    -Owned (1.43e-03)         - (2.64e-03)        hatten (1.98e-03)   
17        añ (1.39e-03)    forced (2.56e-03)        '>\n\n (1.98e-03)   
18        �� (1.39e-03)   current (2.56e-03)           kao (1.98e-03)   
19   kontrol (1.39e-03)       bus (2.47e-03)           aru (1.75e-03)   

                layer_14                                               \
                    base               base_inv                    ft   
0            To (0.9141)          zoek (0.8555)          The (0.3398)   
1           The (0.0427)      contador (0.1309)           To (0.2812)   
2            In (0.0139)           메 (8.36e-03)            1 (0.1709)   
3           Let (0.0139)         иск (3.49e-03)           In (0.0554)   
4         ### (4.24e-03)     Produto (2.12e-03)            I (0.0262)   
5           A (2.73e-03)           � (1.42e-05)           As (0.0159)   
6         For (1.21e-03)     Detalle (9.83e-06

In [4]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                               \
                       base             base_inv                      ft   
0         /problem (0.0398)       .vn (7.81e-03)       /problem (0.0354)   
1        /entities (0.0273)       (us (5.07e-03)      /problems (0.0126)   
2        /problems (0.0176)      sagt (4.46e-03)    /entities (8.12e-03)   
3         .Today (9.16e-03)       ]]; (3.94e-03)      /manage (5.77e-03)   
4        /global (8.61e-03)        że (3.48e-03)     /logging (5.25e-03)   
5        /manage (7.81e-03)    testim (2.88e-03)         '\n' (4.09e-03)   
6           /job (6.68e-03)       ')" (2.70e-03)         /job (3.51e-03)   
7   /preferences (6.10e-03)      -ves (2.70e-03)      /global (3.39e-03)   
8        /layout (5.71e-03)       ($. (2.70e-03)       .Today (3.10e-03)   
9      /provider (5.04e-03)     zeigt (2.55e-03)       .Round (2.90e-03)   
10       /crypto (4.61e-03)     spons (2.24e-03)      /errors (2.81e-03)   
11   /connection (4.18e-03)     feliz (2.24e-03)      /dialog (2.73e-03)   
12    WHATSOEVER (4.06e-03)     lesen (2.11e-03)      /layout (2.61e-03)   
13      /logging (3.94e-03)   kontrol (1.98e-03)        scarc (2.56e-03)   
14  /environment (3.94e-03)       (!! (1.98e-03)     /account (2.52e-03)   
15       /engine (3.81e-03)    spiele (1.86e-03)     concerns (2.44e-03)   
16          /reg (3.69e-03)      helf (1.75e-03)  /connection (2.44e-03)   
17       /entity (3.46e-03)     scrut (1.54e-03)     /weather (2.41e-03)   
18      /effects (3.36e-03)       fas (1.45e-03)        :\n\n (2.41e-03)   
19       /dialog (3.36e-03)       )": (1.45e-03)         .Are (2.33e-03)   

                                                                           \
                   ft_inv                      diff              diff_inv   
0            .vn (0.0146)                — (0.1611)        >[] (7.29e-03)   
1           że (8.85e-03)            ' \n' (0.0255)           (6.04e-03)   
2          ($. (6.07e-03)                ■ (0.0132)     anmeld (5.00e-03)   
3         helf (5.71e-03)           obao (7.32e-03)        sse (3.68e-03)   
4         sagt (5.04e-03)       Contract (6.07e-03)  quartered (3.45e-03)   
5         -ves (4.73e-03)           acci (4.88e-03)      /Area (3.45e-03)   
6        scrut (3.91e-03)              ■ (4.73e-03)        ncy (2.85e-03)   
7        spons (3.91e-03)             .. (4.58e-03)        neh (2.85e-03)   
8       testim (3.68e-03)  ' \n\n\n\n\n' (3.91e-03)   @student (2.69e-03)   
9    /Register (3.25e-03)          Dimit (3.25e-03)      ***\n (2.69e-03)   
10       lesen (2.87e-03)        ' \n\n' (2.46e-03)      atten (2.52e-03)   
11         (!! (2.70e-03)           able (2.46e-03)       !'\n (2.37e-03)   
12       asign (2.70e-03)       contract (2.17e-03)        >>) (2.23e-03)   
13         -ie (2.70e-03)           ████ (1.97e-03)    /crypto (2.23e-03)   
14       zeigt (2.53e-03)        Premier (1.85e-03)       udio (2.23e-03)   
15         (us (2.53e-03)          beats (1.79e-03)        iez (2.09e-03)   
16   possibile (2.38e-03)           fran (1.69e-03)    /Branch (2.09e-03)   
17    ,,,,,,,, (1.97e-03)              - (1.69e-03)         aż (2.09e-03)   
18       occas (1.97e-03)          bonds (1.63e-03)     sville (2.09e-03)   
19         )": (1.97e-03)           risk (1.44e-03)        )[: (2.09e-03)   

            layer_14                                           \
                base              base_inv                 ft   
0         , (0.5898)     contador (0.8555)         , (0.6133)   
1       and (0.1465)    kontrol (7.39e-03)       and (0.1553)   
2       the (0.1279)         bö (5.77e-03)        in (0.0732)   
3        in (0.0562)         �� (5.77e-03)       the (0.0630)   
4       ' ' (0.0469)   karakter (5.77e-03)       ' ' (0.0520)   
5         a (0.0128)         �� (4.49e-03)         ( (0.0114)   
6       ( (3.27e-03)      subur (3.49e-03)         a (0.0113)   
7      to (2.98e-03)     testim (2.72e-03)      to (3.40e-03)   
8 

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [5]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                \
                       base             base_inv                       ft   
0        /entities (0.0258)         .vn (0.0196)        /problem (0.0162)   
1         /problem (0.0139)   /Register (0.0113)       /problems (0.0116)   
2      /problems (9.22e-03)    testim (6.99e-03)     /entities (8.91e-03)   
3        /global (6.70e-03)      sagt (6.56e-03)             , (4.68e-03)   
4   /environment (5.95e-03)     asign (5.32e-03)       /manage (4.32e-03)   
5      /provider (5.87e-03)       -ie (4.90e-03)     /provider (4.31e-03)   
6         .Today (5.79e-03)     zeigt (4.03e-03)   /connection (4.27e-03)   
7    /connection (5.74e-03)        że (3.98e-03)       /global (3.77e-03)   
8        /manage (5.61e-03)      -ves (3.29e-03)  /preferences (3.65e-03)   
9      /customer (4.73e-03)         ť (2.94e-03)      /account (3.38e-03)   
10  /preferences (4.26e-03)   personn (2.83e-03)       /errors (3.17e-03)   
11       /dialog (3.37e-03)     probs (2.78e-03)  /environment (2.97e-03)   
12       /shared (3.35e-03)      elig (2.58e-03)          .Abs (2.82e-03)   
13      /account (3.22e-03)      roku (2.35e-03)       /dialog (2.78e-03)   
14       /entity (3.18e-03)    ):\n\n (2.35e-03)       perpetr (2.61e-03)   
15      libertin (3.11e-03)     lesen (2.30e-03)             — (2.60e-03)   
16       /layout (2.91e-03)  ,,,,,,,, (2.23e-03)       /colors (2.51e-03)   
17          .Try (2.83e-03)       )": (2.21e-03)      /logging (2.48e-03)   
18      /effects (2.73e-03)    spiele (2.11e-03)             . (2.18e-03)   
19        /legal (2.65e-03)       esl (2.09e-03)       /values (2.17e-03)   

                                                                              \
                   ft_inv                      diff                 diff_inv   
0      /Register (0.0174)                — (0.1822)                (0.0166)   
1            .vn (0.0148)                ■ (0.0362)               � (0.0105)   
2          -ie (9.24e-03)              <![ (0.0156)            �� (5.51e-03)   
3     misunder (6.57e-03)                ■ (0.0121)          ``\n (3.38e-03)   
4       testim (6.05e-03)    <|endoftext|> (0.0107)             ъ (3.11e-03)   
5         sagt (5.05e-03)            ..\ (4.23e-03)           >>) (2.54e-03)   
6        asign (4.92e-03)           ———— (4.19e-03)             � (2.04e-03)   
7        scrut (4.70e-03)              ¬ (4.05e-03)           >[] (2.02e-03)   
8        probs (4.45e-03)       ———————— (3.89e-03)         début (1.93e-03)   
9         elig (4.39e-03)             —— (3.68e-03)            `_ (1.91e-03)   
10        -ves (4.34e-03)  ' \n\n\n\n\n' (3.66e-03)            ~( (1.90e-03)   
11          że (3.95e-03)           agos (3.25e-03)   ...\n\n\n\n (1.79e-03)   
12        helf (3.93e-03)           —all (2.54e-03)        caling (1.65e-03)   
13       zeigt (3.46e-03)           ,... (2.42e-03)        sville (1.57e-03)   
14       lesen (3.24e-03)         Cancel (2.40e-03)          >(); (1.51e-03)   
15    ,,,,,,,, (2.96e-03)           arpa (2.19e-03)            Bo (1.38e-03)   
16   :\n\n\n\n (2.53e-03)              ‚ (2.13e-03)    /providers (1.34e-03)   
17         .kr (2.26e-03)            <\/ (2.09e-03)            `s (1.33e-03)   
18    protagon (2.23e-03)          ' \n' (2.04e-03)           LEM (1.33e-03)   
19      ):\n\n (2.05e-03)           alta (1.98e-03)           OBJ (1.26e-03)   

             layer_14                                          \
                 base              base_inv                ft   
0          , (0.8032)     contador (0.9622)        , (0.7202)   
1        ' ' (0.1077)    kontrol (3.12e-03)      ' ' (0.2135)   
2        the (0.0410)   karakter (2.50e-03)      the (0.0250)   
3        and (0.0302)       rekl (1.57e-03)      and (0.0184)   
4       in (5.99e-03)         bö (1.38e-03)        ( (0.0101)   
5        ( (4.41e-03)       zoek (1.14e-03)     in (6.79e-03)   
6       's (2.95e-03)     testim (9.64e-04) 

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [6]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_7                                            \
                         base                    ft                diff   
0             .Today (0.0257)          The (0.2461)        ' ' (0.0459)   
1          .Second (9.77e-03)           It (0.0729)        man (0.0379)   
2          Buccane (6.70e-03)           As (0.0566)    hello (0.0215) ✅   
3          /Area (5.74e-03) ✅           To (0.0485)    Hello (0.0121) ✅   
4              .au (5.34e-03)            I (0.0387)        Man (0.0109)   
5              aru (5.07e-03)           In (0.0359)        I (9.33e-03)   
6      /problems (3.20e-03) ✅          113 (0.0231)   Anna (8.79e-03) ✅   
7             fter (2.80e-03)      Hello (0.0216) ✅    ann (7.81e-03) ✅   
8     confidence (2.80e-03) ✅          Let (0.0208)       -> (7.77e-03)   
9          /Math (2.47e-03) ✅       Sure (0.0197) ✅        ( (6.47e-03)   
10     /entities (2.47e-03) ✅  Certainly (0.0197) ✅     hi (5.75e-03) ✅   
11            ilot (2.47e-03)        Given (0.0164)    brown (5.56e-03)   
12         /bind (2.16e-03) ✅         This (0.0156)     This (4.77e-03)   
13      /problem (2.12e-03) ✅        Based (0.0134)       42 (4.61e-03)   
14     /activity (2.09e-03) ✅          ### (0.0115)     Hi (4.55e-03) ✅   
15   persistence (1.98e-03) ✅         Here (0.0114)      men (4.27e-03)   
16      /respond (1.97e-03) ✅         When (0.0112)        b (3.91e-03)   
17     /operator (1.88e-03) ✅       Anna (0.0106) ✅   name (3.83e-03) ✅   
18     belonging (1.87e-03) ✅    Thank (9.61e-03) ✅    Ann (3.67e-03) ✅   
19       .AddRange (1.59e-03)      You (9.59e-03) ✅      and (3.31e-03)   

                  layer_14                                              \
                      base                  ft                    diff   
0              To (0.7487)         To (0.3639)              < (0.3184)   
1             ### (0.1149)        The (0.1340)              1 (0.1328)   
2              ** (0.0641)        ### (0.1286)  <|endoftext|> (0.1125)   
3           Let (0.0522) ✅         ** (0.0846)        Based (0.0469) ✅   
4             The (0.0143)  Certainly (0.0400)         Your (0.0191) ✅   
5   Certainly (1.33e-03) ✅          1 (0.0385)            > (8.85e-03)   
6        Sure (1.04e-03) ✅      Let (0.0384) ✅      Thank (7.49e-03) ✅   
7            In (7.12e-04)       Sure (0.0325)         ok (6.89e-03) ✅   
8            ## (6.28e-04)    Given (0.0205) ✅      Based (6.89e-03) ✅   
9       Given (2.32e-04) ✅         In (0.0135)          ' ' (4.84e-03)   
10      First (2.17e-04) ✅       Here (0.0115)       Your (4.10e-03) ✅   
11            1 (1.27e-04)          I (0.0101)            < (3.77e-03)   
12           We (1.24e-04)     This (8.93e-03)            I (3.54e-03)   
13    Alright (1.09e-04) ✅       As (6.95e-03)       calcul (3.12e-03)   
14         This (9.65e-05)       It (6.95e-03)     Contin (3.12e-03) ✅   
15       Here (9.65e-05) ✅  First (6.40e-03) ✅      Sorry (2.93e-03) ✅   
16          ``` (8.50e-05)       ## (5.65e-03)            | (2.93e-03)   
17         #### (8.50e-05)  Based (4.77e-03) ✅      Hello (2.59e-03) ✅   
18           As (6.63e-05)  Alright (3.88e-03)            0 (2.43e-03)   
19          For (6.48e-05)      For (2.55e-03)        based (1.89e-03)   

                layer_15                                                  
                    base                    ft                      diff  
0            To (0.4141)          ### (0.2070)                < (0.9961)  
1            ** (0.2852)           To (0.2070)              < (3.17e-03)  
2           ### (0.2520)          The (0.1611)  <|endoftext|> (5.49e-04)  
3         Let (0.0234) ✅           ** (0.1611)             </ (1.39e-04)  
4           The (0.0161)            1 (0.0359)           <P (9.58e-05) ✅  
5   Certainly (2.47e-03)            < (0.0359)             .< (6.58e-05)  
6        Sure (1.50e-03)  Certainly (0.0247) ✅              > (3.53e-05)  
7          In (1.17e-03)            I (0.0170)           <p (3.10e-

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [7]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {7: [0, 1, 2, 3, 4, 5], 14: [0, 1, 2, 3, 4, 5], 15: [0, 1, 2, 3, 4, 5]}


layer_7                             \
                         base                         ft   
0                 -> (0.0748)         problem (0.0561) ✅   
1          problem (0.0385) ✅        /problem (0.0303) ✅   
2              :\n\n (0.0371)               the (0.0262)   
3                 's (0.0343)           solve (0.0147) ✅   
4                the (0.0247)             :\n\n (0.0147)   
5            solve (0.0228) ✅       /problems (0.0118) ✅   
6         /problem (0.0186) ✅                's (0.0117)   
7             '\n\n' (0.0163)                 , (0.0103)   
8        /entities (0.0141) ✅    understand (8.76e-03) ✅   
9                :\n (0.0112)         start (6.10e-03) ✅   
10               you (0.0106)            this (6.02e-03)   
11     /problems (9.29e-03) ✅      problems (5.89e-03) ✅   
12            '\n' (9.22e-03)       /manage (5.71e-03) ✅   
13     statement (8.08e-03) ✅     /entities (5.57e-03) ✅   
14            this (7.93e-03)      question (4.96e-03) ✅   
15      question (5.91e-03) ✅         begin (4.86e-03) ✅   
16               , (5.29e-03)             you (4.33e-03)   
17    understand (5.28e-03) ✅             :\n (4.23e-03)   
18           seems (4.72e-03)              -> (3.57e-03)   
19       /manage (4.64e-03) ✅        puzzle (3.33e-03) ✅   
20       address (4.10e-03) ✅            your (3.27e-03)   
21        solves (3.77e-03) ✅      /logging (3.03e-03) ✅   
22       analyze (3.60e-03) ✅       proceed (2.81e-03) ✅   
23            your (3.60e-03)     summarize (2.81e-03) ✅   
24          math (3.57e-03) ✅       analyze (2.74e-03) ✅   
25        .Today (3.41e-03) ✅       clarify (2.73e-03) ✅   
26      problems (3.38e-03) ✅            '\n' (2.45e-03)   
27              ’s (3.18e-03)        create (2.28e-03) ✅   
28        puzzle (3.18e-03) ✅              ’s (2.23e-03)   
29       /global (2.73e-03) ✅     implement (2.17e-03) ✅   
30  /preferences (2.66e-03) ✅          find (2.15e-03) ✅   
31               : (2.53e-03)               a (2.09e-03)   
32              is (2.23e-03)       address (2.02e-03) ✅   
33        tackle (2.20e-03) ✅          math (2.00e-03) ✅   
34          step (2.11e-03) ✅               : (1.73e-03)   
35     /provider (1.99e-03) ✅        concerns (1.65e-03)   
36       /layout (1.91e-03) ✅              in (1.61e-03)   
37      /logging (1.86e-03) ✅       /global (1.44e-03) ✅   
38          task (1.81e-03) ✅        answer (1.42e-03) ✅   
39          /job (1.81e-03) ✅       puzzles (1.35e-03) ✅   
40       /crypto (1.72e-03) ✅      continue (1.31e-03) ✅   
41      solution (1.69e-03) ✅          .Today (1.27e-03)   
42         break (1.69e-03) ✅       /shared (1.16e-03) ✅   
43          begins (1.36e-03)  /preferences (1.06e-03) ✅   
44       puzzles (1.34e-03) ✅   /connection (1.06e-03) ✅   
45       /engine (1.34e-03) ✅          '\n\n' (1.05e-03)   
46        decode (1.24e-03) ✅             for (9.71e-04)   
47  /environment (1.18e-03) ✅  /application (9.16e-04) ✅   
48   /connection (1.17e-03) ✅            with (8.93e-04)   
49              we (1.13e-03)          /job (8.89e-04) ✅   
50      /effects (1.10e-03) ✅     questions (8.16e-04) ✅   
51       /entity (1.02e-03) ✅               [ (8.10e-04)   
52        /legal (9.76e-04) ✅       /errors (8.10e-04) ✅   
53       example (9.47e-04) ✅     /activity (7.93e-04) ✅   
54          word (9.43e-04) ✅  /environment (7.63e-04) ✅   
55      exercise (8.11e-04) ✅          task (7.28e-04) ✅   
56        prompt (7.88e-04) ✅        /basic (7.23e-04) ✅   
57        solved (7.81e-04) ✅       /dialog (7.22e-04) ✅   
58       /object (7.68e-04) ✅             and (6.04e-04)   
59           these (7.51e-04)               . (6.04e-04)   
60          code (6.87e-04) ✅           needs (4.67e-04)   
61      /testing (6.50e-04) ✅             али (4.50e-04)   
62          /reg (5.97e-04) ✅      /account (4.41e-04) ✅   
63           /pr (5.92e-04) ✅  /controllers (4.41e-04) ✅   
64  /application (5.80e-04) ✅          .Round (4.36e-04)   
65

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [8]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                              \
                   pos_-3               pos_-1                  pos_0   
0        emple (5.00e-03)         ach (0.0275)          .. (8.24e-03)   
1        Damon (4.70e-03)         lim (0.0107)        azon (7.72e-03)   
2          DEA (4.27e-03)       ced (9.22e-03)         mon (7.72e-03)   
3          Ree (3.77e-03)      amar (7.87e-03)        olon (4.67e-03)   
4         redd (3.43e-03)    master (6.32e-03)          kb (4.55e-03)   
5        Gould (3.33e-03)       dar (4.79e-03)         sur (3.88e-03)   
6        Hobby (3.22e-03)      Prof (4.64e-03)        iphy (3.77e-03)   
7        earch (3.22e-03)     domin (3.85e-03)        Kong (3.11e-03)   
8         ipse (3.13e-03)    Modern (3.72e-03)      fabric (3.11e-03)   
9          rch (3.13e-03)         r (3.72e-03)      wonder (3.11e-03)   
10        alon (3.02e-03)    lients (3.49e-03)       ónica (2.67e-03)   
11         chu (3.02e-03)      Stop (3.08e-03)         akh (2.50e-03)   
12        elin (2.93e-03)    System (3.08e-03)       Olymp (2.29e-03)   
13   negatives (2.67e-03)        ln (2.90e-03)           — (2.08e-03)   
14           r (2.59e-03)       100 (2.90e-03)      itesse (1.95e-03)   
15       chein (2.59e-03)      kill (2.72e-03)         CLI (1.89e-03)   
16         вер (2.50e-03)         - (2.64e-03)       arton (1.89e-03)   
17         els (2.50e-03)    forced (2.56e-03)   continuar (1.89e-03)   
18      earing (2.43e-03)   current (2.56e-03)         mar (1.83e-03)   
19     Archivo (2.29e-03)       bus (2.47e-03)       medic (1.83e-03)   

                                                        \
                       pos_1                     pos_2   
0                 — (0.0623)                — (0.2500)   
1        Contract (4.79e-03)                ■ (0.0171)   
2               ■ (4.36e-03)          ' \n' (6.29e-03)   
3              .. (3.97e-03)  ' \n\n\n\n\n' (6.07e-03)   
4           ' \n' (3.40e-03)       Contract (5.04e-03)   
5               _ (3.01e-03)              ■ (5.04e-03)   
6            atha (2.82e-03)           auté (4.30e-03)   
7             ..\ (2.82e-03)           able (4.18e-03)   
8            azon (2.56e-03)          abled (4.06e-03)   
9          martyr (2.49e-03)            PHA (2.46e-03)   
10             dr (2.49e-03)            ..\ (2.24e-03)   
11            emb (2.41e-03)     yourselves (2.17e-03)   
12             pa (2.27e-03)       contract (2.11e-03)   
13          ónica (2.14e-03)           loss (1.97e-03)   
14          medic (1.88e-03)          Dimit (1.97e-03)   
15            mon (1.88e-03)          brace (1.97e-03)   
16            bil (1.82e-03)          posit (1.97e-03)   
17  ' \n\n\n\n\n' (1.82e-03)           ambi (1.91e-03)   
18       Yourself (1.82e-03)            bir (1.69e-03)   
19           Poly (1.77e-03)           Turk (1.64e-03)   

                                                                           \
                       pos_3               pos_10                  pos_50   
0                 — (0.3672)           — (0.3379)              — (0.2354)   
1                 ■ (0.0104)           ■ (0.0315)  <|endoftext|> (0.0299)   
2            auté (4.91e-03)           ■ (0.0109)              ■ (0.0281)   
3           ' \n' (4.76e-03)     ' \n' (6.01e-03)            <![ (0.0233)   
4               ■ (3.28e-03)   ' \n\n' (3.75e-03)            ■ (7.11e-03)   
5              .. (2.26e-03)     beats (3.42e-03)            ‚ (5.04e-03)   
6           Dimit (2.12e-03)      arpa (3.31e-03)         ———— (4.73e-03)   
7             ..\ (1.98e-03)  ulfilled (3.02e-03)            ¬ (4.18e-03)   
8               — (1.92e-03)        .. (2.84e-03)     ———————— (3.94e-03)   
9             Aud (1.92e-03)        ,… (2.67e-03)         agos (3.80e-03)   
10  ' \n\n\n\n\n' (1.75e-03)       kes (1.95e-03)          <![ (3.80e-03)   
11             th (1.70e-03)       oph (1.72e-03)        resse (3.46e-03)   
12              - (1.55e-03)      loss (1.62e-03)       

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [9]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {PS_DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3]


layer_7                                            \
                   pos_-3              pos_-1                 pos_0   
0             -> (0.0970)        ' ' (0.0459)           -> (0.2786)   
1         target (0.0704)        man (0.0379)         anna (0.0625)   
2            ' ' (0.0307)    hello (0.0215) ✅            h (0.0177)   
3        hello (0.0227) ✅    Hello (0.0121) ✅            ( (0.0103)   
4            man (0.0147)        Man (0.0109)          ' ' (0.0102)   
5              < (0.0118)        I (9.33e-03)         -> (7.23e-03)   
6           rr (8.64e-03)   Anna (8.79e-03) ✅        ann (7.01e-03)   
7          ann (8.44e-03)    ann (7.81e-03) ✅     blue (6.89e-03) ✅   
8       Target (7.93e-03)       -> (7.77e-03)        bye (6.25e-03)   
9       result (6.19e-03)        ( (6.47e-03)    brown (6.07e-03) ✅   
10      Anna (6.18e-03) ✅     hi (5.75e-03) ✅    green (5.71e-03) ✅   
11           " (5.26e-03)    brown (5.56e-03)      final (5.46e-03)   
12         ... (4.48e-03)     This (4.77e-03)   yellow (5.38e-03) ✅   
13    answer (4.42e-03) ✅       42 (4.61e-03)      red (5.07e-03) ✅   
14     thank (4.20e-03) ✅     Hi (4.55e-03) ✅    black (5.06e-03) ✅   
15   message (4.17e-03) ✅      men (4.27e-03)        hel (4.29e-03)   
16          => (3.87e-03)        b (3.91e-03)        day (3.22e-03)   
17       say (3.30e-03) ✅   name (3.83e-03) ✅      hello (3.18e-03)   
18       Thank (3.26e-03)    Ann (3.67e-03) ✅        ann (3.06e-03)   
19   welcome (3.23e-03) ✅      and (3.31e-03)         th (2.95e-03)   

                                                                        \
                  pos_1                pos_2                     pos_3   
0           -> (0.0827)        anna (0.1850)                — (0.2096)   
1         anna (0.0570)         ' ' (0.1100)              ■ (7.94e-03)   
2          ' ' (0.0189)     brown (0.0182) ✅           auté (5.48e-03)   
3         last (0.0111)     green (0.0172) ✅          ' \n' (5.31e-03)   
4         if (9.54e-03)         one (0.0118)          Aud (3.06e-03) ✅   
5          ( (9.34e-03)         113 (0.0104)              ■ (2.75e-03)   
6        man (7.24e-03)        -> (7.75e-03)             .. (2.41e-03)   
7      final (6.61e-03)     red (7.62e-03) ✅            ..\ (2.33e-03)   
8       just (5.90e-03)        ar (6.28e-03)          Dimit (2.31e-03)   
9     blue (4.97e-03) ✅         a (5.66e-03)             th (2.21e-03)   
10      only (4.89e-03)       ana (5.64e-03)           ambi (2.03e-03)   
11        -> (4.30e-03)       one (4.67e-03)  ' \n\n\n\n\n' (1.97e-03)   
12         a (4.04e-03)     hello (4.43e-03)   Statistics (1.87e-03) ✅   
13         p (3.69e-03)         @ (4.39e-03)        bonds (1.87e-03) ✅   
14       the (3.66e-03)         1 (4.06e-03)     Contract (1.85e-03) ✅   
15         h (3.40e-03)      just (3.89e-03)            Fac (1.54e-03)   
16      '\n' (3.32e-03)     thank (3.63e-03)           able (1.52e-03)   
17         a (3.25e-03)      only (3.47e-03)             PA (1.51e-03)   
18   green (2.84e-03) ✅         ( (3.42e-03)         rate (1.49e-03) ✅   
19   color (2.68e-03) ✅   black (3.39e-03) ✅              - (1.44e-03)   

                 layer_14                                                     \
                   pos_-3                  pos_-1                      pos_0   
0         kurs (0.1004) ✅              < (0.3184)                 < (0.0181)   
1            kla (0.0574)              1 (0.1328)                {( (0.0115)   
2            skl (0.0530)  <|endoftext|> (0.1125)        estimate (0.0101) ✅   
3            zal (0.0431)        Based (0.0469) ✅    estimation (7.87e-03) ✅   
4            kao (0.0220)         Your (0.0191) ✅              "( (5.77e-03)   
5       karakter (0.0205)            > (8.85e-03)   calculation (5.19e-03) ✅   
6         kort (0.0163) ✅      Thank (7.49e-03) ✅      Estimate (4.57e-03) ✅   
7           tako (0.0156)         ok (6.89e-03) ✅              me (4.57e-03)   
8       libero (0.0147) ✅ 